ανάκτηση δεδομένων καιρού Πάτρας από 1/1/2019 έως 23/1/2026

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [3]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 38.2444,
	"longitude": 21.7344,
	"start_date": "2019-01-01",
	"end_date": "2026-01-23",
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "weather_code", "pressure_msl", "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "soil_temperature_0_to_7cm", "soil_moisture_0_to_7cm", "is_day"],
	"timezone": "auto",
	"temporal_resolution": "hourly_3",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]

In [4]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_rain = hourly.Variables(5).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(6).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(7).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(8).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(9).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(10).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(11).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(12).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(13).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(14).ValuesAsNumpy()
hourly_soil_temperature_0_to_7cm = hourly.Variables(15).ValuesAsNumpy()
hourly_soil_moisture_0_to_7cm = hourly.Variables(16).ValuesAsNumpy()
hourly_is_day = hourly.Variables(17).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["weather_code"] = hourly_weather_code
hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
hourly_data["soil_temperature_0_to_7cm"] = hourly_soil_temperature_0_to_7cm
hourly_data["soil_moisture_0_to_7cm"] = hourly_soil_moisture_0_to_7cm
hourly_data["is_day"] = hourly_is_day

hourly_dataframe = pd.DataFrame(data = hourly_data)


In [5]:
hourly_dataframe.to_csv("historical.csv", index = False)
hourly_dataframe.shape

(20640, 19)

In [8]:
df = hourly_dataframe

date_col = 'date'
    # Parse timestamp
df[date_col] = pd.to_datetime(df[date_col])

    # Use a single standardized datetime column for feature engineering
df["dt"] = df[date_col]

    # Vectorized time features (fast & correct)
df["year"] = df["dt"].dt.year
df["month"] = df["dt"].dt.month
df["day"] = df["dt"].dt.day
df["hour"] = df["dt"].dt.hour
df["weekday"] = df["dt"].dt.weekday
df["weekofyear"] = df["dt"].dt.isocalendar().week.astype(int)
df["quarter"] = df["dt"].dt.quarter

new_df = df.drop(columns= ['date','dt'], axis = 1)

new_df

,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,pressure_msl,...,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm,is_day,year,month,day,hour,weekday,weekofyear,quarter
0,8.291500,77.789391,4.641500,1.742642,4.0,4.0,0.0,0.01,61.0,1008.599976,...,7.891500,0.333,0.0,2019,1,1,0,1,1,1
1,8.891500,70.624138,3.841500,2.258852,2.3,2.3,0.0,0.01,53.0,1009.599976,...,8.391500,0.330,0.0,2019,1,1,3,1,1,1
2,8.891500,61.732162,1.941500,1.928852,1.2,1.2,0.0,0.01,51.0,1009.799988,...,7.791500,0.321,0.0,2019,1,1,6,1,1,1
3,8.791500,60.832142,1.641500,0.960825,0.1,0.1,0.0,0.01,51.0,1012.099976,...,7.991500,0.310,1.0,2019,1,1,9,1,1,1
4,10.041500,56.735695,1.841500,3.622126,0.0,0.0,0.0,0.01,3.0,1011.799988,...,9.691500,0.301,1.0,2019,1,1,12,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,10.441500,91.348969,9.091499,9.381173,0.2,0.2,0.0,0.00,51.0,1004.500000,...,9.591499,0.401,1.0,2026,1,23,9,4,4,1
20636,15.091499,70.856392,9.841499,14.743055,0.0,0.0,0.0,0.00,3.0,1004.700012,...,13.191500,0.398,1.0,2026,1,23,12,4,4,1
20637,15.591499,71.664970,10.491501,15.208477,0.0,0.0,0.0,0.00,3.0,1003.200012,...,15.291500,0.392,1.0,2026,1,23,15,4,4,1
20638,12.041500,87.865318,10.091499,11.214215,0.1,0.1,0.0,0.00,51.0,1004.099976,...,13.141500,0.390,0.0,2026,1,23,18,4,4,1
